# 03 · Foundation models with a uniform context + matched-budget XGBoost (tasks T5, T6)

**T5.** TabPFN v2 and TabICLv2 at every fraud rate and every context budget k ∈ {1K, 2K, 5K, 10K} that fits the training pool, 5 seeds. The context is k training rows drawn at random ("uniform"), the naive baseline that W1 will try to beat.

**T6.** XGBoost trained on *exactly the same k rows*. This separates the model family from the amount of data: full-data XGBoost (notebook 02) sees all ~170K training rows, while a TFM sees only k.

**Time on a T4** (measured in the smoke test): TabICLv2 takes about 20 minutes for everything. TabPFN v2 takes about 27 minutes per seed, so run it a seed or two per session. Everything is saved as it goes. Set `SEEDS` below, **Run all**, and rerun with the next seeds in a later session; finished cells are skipped.

In [ ]:
# 1. What to run in this session. Suggested plan:
#    session 1: MODELS = ["TabICLv2", "TabPFN v2"], SEEDS = [0]
#    session 2: SEEDS = [1, 2]    session 3: SEEDS = [3, 4]
#    (TabICLv2 is fast; it finishes its seeds in each session alongside TabPFN.)
MODELS = ["TabICLv2", "TabPFN v2"]
SEEDS = [0]
RUN_MATCHED_XGB = True   # T6; fast, runs for the same SEEDS

In [ ]:
# 2. Drive, code, libraries (same as notebook 02).
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 3. Unit tests, then data and splits with the fingerprint check.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q
from fraudicl.config import DRIVE_DIR, CONTEXT_BUDGETS
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
print("splits OK", expected)

In [ ]:
# 3b. One-time clean-up (20 Oct): the first seed-0 run drew each context size k
#     independently. Contexts are now nested across k, so those runs are moved to
#     results/archive_v1_independent_contexts/ on Drive and redone. Nothing is
#     deleted, full-data XGBoost is untouched, and after the first session this
#     cell does nothing.
from fraudicl.experiments import archive_old_uniform_runs
OUT = f"{DRIVE_DIR}/results"
print("moved old runs to archive" if archive_old_uniform_runs(OUT) else "nothing to archive")

In [ ]:
# 4. T5: the foundation models with a uniform context.
#    Each line shows how many frauds the context held, the test PR-AUC and the time.
from fraudicl.models import make_tabpfn_v2, make_tabicl_v2
from fraudicl.experiments import run_tfm_sweep
OUT = f"{DRIVE_DIR}/results"
LEVELS = ["20", "5", "1", "0.5", "natural"]
makers = {"TabICLv2": make_tabicl_v2, "TabPFN v2": make_tabpfn_v2}
run_tfm_sweep(X, y, splits, OUT, {m: makers[m] for m in MODELS},
              levels=LEVELS, budgets=CONTEXT_BUDGETS, seeds=SEEDS, policy="uniform")

In [ ]:
# 5. T6: XGBoost on exactly the same k rows (plain variant, 50 trials per cell on seed 0).
from fraudicl.experiments import run_matched_xgb
if RUN_MATCHED_XGB:
    run_matched_xgb(X, y, splits, OUT, levels=LEVELS, budgets=CONTEXT_BUDGETS,
                    seeds=SEEDS, policy="uniform", variant="plain", n_trials=50)

In [ ]:
# 6. Results so far, mean over the seeds finished: one row per model, fraud rate and k.
import pandas as pd
from fraudicl.experiments import collect, summarise_by_k
parts = [collect(OUT, "tfm")]
if os.path.exists(f"{OUT}/raw/xgb_matched"):
    parts.append(collect(OUT, "xgb_matched"))
by_k = summarise_by_k(pd.concat(parts, ignore_index=True))
by_k.round(4)

**How to read this table.**

- `context_frauds_mean` is how many frauds the model actually saw. At the natural rate with k = 1K it's under 2, and `seeds_with_no_fraud` counts seeds where it was zero. Those runs score exactly the fraud rate (no skill).
- Compare a TFM row with the `XGBoost (plain, k rows)` row at the same fraud rate and k: same rows, different model family. That is the fair comparison.
- Compare either one with full-data XGBoost from notebook 02 (natural PR-AUC 0.864). The gap is the price of seeing only k rows.
- PR-AUC should rise with k, mostly because a bigger uniform context holds more frauds. W1 tests whether choosing the rows cleverly gets the same gain without needing a bigger k.

In [ ]:
# 7. Paste this back to Claude.
print(by_k.round(4).to_csv(index=False))